# 第8章 正则表达式

正则表达式是字符串处理的有力工具,使用预定义的模式去匹配一类具有共同特征的字符串,可以快速、准确地完成复杂的查找、替换等处理要求。在文本编辑与处理、网页爬虫之类的场合中有重要应用。

## 8.1 正则表达式语法

| 元字符 | 功能说明 |
|---|---|
| `.` | 匹配除换行符以外的任意单个字符 |
| `*`, `+` | 匹配位于之前的字符或子模式的0次/多次或1次/多次出现 |
| `^`, `$` | 匹配行首、匹配行尾 |
| `?` | 匹配位于之前的0个或1个字符。加在限定符后代表“非贪心”模式 |
| `\b`, `\B` | 匹配单词头或尾, `\B`含义相反 |
| `\d`, `\D` | 匹配任何数字(`[0-9]`), `\D`含义相反 |
| `\s`, `\S` | 匹配任何空白字符, `\S`含义相反 |
| `\w`, `\W` | 匹配任何字母、数字以及下划线, `\W`含义相反 |
| `()`, `[]` | 作为一个整体对待(子模式); 表示范围 |
| `{m, n}` | 重复至少m次,至多n次 |

### 扩展语法:
| 语法 | 功能说明 |
|---|---|
| `(?P<groupname>)` | 为子模式命名 |
| `(?:...)` | 匹配但不捕获该匹配的子表达式 |
| `(?=...)`, `(?!...)` | 正向肯定预查、正向否定预查 (判断之后的内容) |
| `(?<=...)`, `(?<!...)` | 反向肯定预查、反向否定预查 (判断之前的内容) |

## 8.2 直接使用正则表达式模块re处理字符串

常用函数:
* `compile(pattern[, flags])`: 创建模式对象
* `findall(pattern, string)`: 返回所有匹配项的列表
* `match(pattern, string)`: 从开始处匹配
* `search(pattern, string)`: 在整个字符串寻找
* `split(pattern, string)`: 分隔字符串
* `sub(pat, repl, string)`: 替换匹配的项

In [ ]:
import re
text = 'alpha. beta....gamma delta'
print(re.split('[\\. ]+', text))
print(re.split('[\\. ]+', text, maxsplit=1))

pat = '[a-zA-Z]+'
print(re.findall(pat, text))

# 字符串替换
pat2 = '{name}'
text2 = 'Dear {name}...'
print(re.sub(pat2, 'Mr.Dong', text2))

s = "It's a very good good idea"
print(re.sub(r'(\b\w+) \1', r'\1', s)) # 处理重复单词

# 使用lambda进行高级替换
print(re.sub('a', lambda x: x.group(0).upper(), 'aaa abc abde'))

# match与search的区别
print(re.match('done|quit', 'done')) # 返回 match object
print(re.match('done|quit', 'd!one!')) # 返回 None
print(re.search('done|quit', 'd!one!done')) # 返回 match object

In [ ]:
import re
# 多种去除多余空格的方法比较
s = 'aaa      bb      cde   fff'
print(' '.join(s.split()))
print(' '.join(re.split('\s+', s.strip())))
print(re.sub('\s+', ' ', s.strip()))

# 贪心与非贪心模式
example = 'Beautiful is better than ugly.'
print(re.findall('\\bb.+?\\b', example)) # 非贪心
print(re.findall('\\bb.+\\b', example)) # 贪心

s = '<html><head>This is head.</head><body>This is body.</body></html>'
pattern = r'<html><head>(.+)</head><body>(.+)</body></html>'
result = re.search(pattern, s)
print(result.group(1))
print(result.group(2))

## 8.3 使用正则表达式对象处理字符串
首先使用re模块的`compile()`方法将正则表达式编译生成正则表达式对象,然后再使用正则表达式对象提供的方法进行字符串处理。可以提高处理速度和功能复杂度。

In [ ]:
import re
example = 'ShanDong Institute of Business and Technology'
pattern = re.compile(r'\bB\w+\b')
print(pattern.findall(example)) # ['Business']

pattern = re.compile(r'\w+g\b')
print(pattern.findall(example)) # ['ShanDong']

s = 'ab134ab98723jafjweoruiagab'
m = re.search(r'((ab).*){2}.*(ab)', s)
print(m.group(3)) # 'ab'
print(m.span(3)) # (24, 26)

example_str = '''Beautiful is better than ugly.
Explicit is better than implicit.'''
pattern = re.compile(r'\bb\w*\b', re.I)
print(pattern.sub('*', example_str, 2)) # 将符合条件的单词替换为*,只替换前2次

## 8.4 match对象
正则表达式对象的`match`方法和`search`方法匹配成功后返回match对象。
* `group()`: 返回匹配的一个或多个子模式内容
* `groups()`: 返回一个包含匹配的所有子模式内容的元组
* `groupdict()`: 返回字典形式
* `start()`, `end()`, `span()`: 获取位置信息

In [ ]:
m = re.match(r"(\w+) (\w+)", "Isaac Newton, physicist")
print(m.group(0)) # 'Isaac Newton'
print(m.group(1, 2)) # ('Isaac', 'Newton')

m = re.match(r"(?P<first_name>\w+) (?P<last_name>\w+)", "Malcolm Reynolds")
print(m.group('first_name')) # 'Malcolm'
print(m.groupdict()) # {'first_name': 'Malcolm', 'last_name': 'Reynolds'}

# 结合预查的高级查找
exampleString = 'There should be one-- and preferably only one obvious way...'
pattern = re.compile(r'(?<!not\s)be\b') # 查找前面没有单词not的单词be
matchResult = pattern.search(exampleString)
if matchResult:
    print(matchResult.group(0), ':', matchResult.span(0))

## 8.5 精彩案例赏析
使用正则表达式提取字符串中的电话号码；批量检查网页文件是否被嵌入iframe框架。

In [ ]:
import re
telNumber = '''Suppose my Phone No. is 0535-1234567,
yours is 010-12345678,
his is 025-87654321.'''

pattern = re.compile(r'(\d{3,4})-(\d{7,8})')
index = 0
while True:
    matchResult = pattern.search(telNumber, index)
    if not matchResult:
        break
    print('-'*30)
    print('Success:')
    for i in range(3):
        print('Searched content:', matchResult.group(i), \
              'Start from:', matchResult.start(i), 'End at:', matchResult.end(i), \
              'Its span is:', matchResult.span(i))
    index = matchResult.end(2)